# ELIZA kontra LLM — gdzie jest „rozumienie"?

**Podstawy AI · Laboratorium 2**

Na poprzednich zajęciach zbudowałeś ELIZĘ i znalazłeś miejsca, w których
jej złudzenie pęka. Dziś te same zdania trafią do dużego modelu językowego
(LLM) i sprawdzimy: **które pęknięcia LLM naprawia, a które zostają?**

**Plan zajęć**

| Sekcja | Co robisz |
|---|---|
| 1. Jak rozmawia się z LLM przez API | czytasz (5 min) |
| 2. Przygotowanie | klucz, klient, test połączenia |
| 3. Twoja ELIZA wraca | wklejasz kod z Lab 1 |
| 4. Ta sama rozmowa, dwa silniki | Zadanie 8 |
| 5. Łamiące zdania — druga runda | Zadanie 9 |
| 6. Pamięć to lista wiadomości | Zadanie 10 |
| 7. Rola to tekst | Zadanie 11 |
| 8. Refleksja | Zadanie 12 |

**Potrzebujesz:** klucza Gemini API (instrukcję dostałeś przed zajęciami)
i notebooka z Lab 1.

> ⚠️ **Nie wpisuj prawdziwych, osobistych zwierzeń.** W darmowym planie
> dostawca może wykorzystywać przesłane treści do ulepszania usług.
> Rozmawiamy z „terapeutą", ale wszystkie zdania mają być **wymyślone**.

## 1. Jak rozmawia się z LLM przez API

**Czym jest LLM (w pięciu zdaniach).** Duży model językowy to sieć neuronowa
wytrenowana na ogromnych zbiorach tekstu do jednego zadania: przewidywania,
jaki fragment tekstu (*token*) powinien pojawić się dalej. Odpowiedź powstaje
token po tokenie — model za każdym razem wybiera kolejny fragment, a wybór
jest częściowo losowy (*próbkowanie*). Nie ma tu reguł pisanych ręcznie jak
w ELIZIE: wszystko, co model „umie", jest zapisane w miliardach wyuczonych
parametrów. To, czy i w jakim sensie model *rozumie* tekst, jest przedmiotem
sporu — i właśnie dlatego porównujemy go dziś z ELIZĄ.

**Zapytanie do API.** Rozmawiamy z modelem, wysyłając mu **listę wiadomości**.
Każda wiadomość ma rolę:

| Rola | Kto mówi | Przykład |
|---|---|---|
| `system` | instrukcja dla modelu — kim ma być, jak odpowiadać | „Jesteś terapeutą. Odpowiadaj krótko." |
| `user` | użytkownik | „Czuję się samotny." |
| `assistant` | wcześniejsze odpowiedzi modelu | „Od kiedy tak się czujesz?" |

```python
messages = [
    {"role": "system", "content": "Jesteś terapeutą..."},
    {"role": "user",   "content": "Czuję się samotny."},
]
```

W odpowiedzi dostajemy **jedną** nową wiadomość modelu. Zapamiętaj ten
schemat — w Zadaniu 10 okaże się ważniejszy, niż wygląda.

**Biblioteka.** Użyjemy biblioteki `openai`. Mimo nazwy nie łączymy się
z OpenAI: Gemini udostępnia *zgodny* interfejs, więc wystarczy podać inny
adres serwera (`base_url`) i swój klucz.

## 2. Przygotowanie

Klucz podajesz przez `getpass` — nie pojawi się na ekranie ani w zapisanym
notebooku. **Nigdy nie wklejaj klucza wprost do kodu**: notebook oddajesz
prowadzącemu, a klucz to dostęp do Twojego konta.

Jeśli pracujesz w Google Colab, możesz zamiast tego zapisać klucz
w *Secrets* (ikona klucza po lewej) pod nazwą `GEMINI_API_KEY`.

In [1]:
# !pip install openai        # odkomentuj, jeśli biblioteki nie ma (w Colabie jest)
import os
import time
from getpass import getpass
from openai import OpenAI

try:                                    # Colab: klucz z Secrets
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:                       # lokalnie: zmienna środowiskowa albo getpass
    API_KEY = os.environ.get("GEMINI_API_KEY") or getpass("Wklej klucz Gemini API: ")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.5-flash-lite"   # jeśli nie działa: sprawdź listę w następnej komórce
REASONING = "minimal"             # ogranicza „myślenie" modelu -> szybciej i taniej;
                                  # jeśli dostajesz błąd 400 dotyczący reasoning_effort, ustaw None
print("Klient gotowy. Model:", MODEL)

Klient gotowy. Model: gemini-3.5-flash-lite


In [2]:
# Jakie modele Flash są dostępne dla Twojego klucza?
for m in client.models.list():
    if "flash" in m.id:
        print(m.id)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


Poniższa funkcja `czat` wysyła listę wiadomości i zwraca tekst odpowiedzi.
Obsługuje też błąd **429** (przekroczony limit zapytań na minutę) — wtedy
odczekuje chwilę i próbuje ponownie. Nie musisz jej modyfikować.

In [3]:
def czat(messages, temperatura=None, max_tokens=400):
    params = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
    if REASONING:
        params["reasoning_effort"] = REASONING
    if temperatura is not None:
        params["temperature"] = temperatura
    for proba in range(3):
        try:
            r = client.chat.completions.create(**params)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "429" in str(e) and proba < 2:
                print("  (limit zapytań — czekam 20 s)")
                time.sleep(20)
            else:
                raise

SYSTEM_TERAPEUTA = ("Jesteś terapeutą prowadzącym rozmowę po polsku. "
                    "Odpowiadaj krótko, jednym–dwoma zdaniami.")

def zapytaj_llm(tekst, system=SYSTEM_TERAPEUTA):
    # JEDNO pytanie, BEZ historii rozmowy
    return czat([{"role": "system", "content": system},
                 {"role": "user",   "content": tekst}])

print(zapytaj_llm("Czuję się dzisiaj samotny."))   # test połączenia

Przykro mi, że tak się czujesz. Chcesz o tym opowiedzieć?


**Coś nie działa?**

| Objaw | Przyczyna |
|---|---|
| `401` / `API key not valid` | źle skopiowany klucz — uruchom komórkę z kluczem jeszcze raz |
| `404` / `model not found` | nazwa modelu się zmieniła — wybierz z listy wyżej |
| `400` z `reasoning_effort` | ustaw `REASONING = None` i uruchom komórki ponownie |
| `429` | limit zapytań — odczekaj minutę; nie uruchamiaj pętli wielokrotnie |
| pusta odpowiedź `""` | model zużył limit tokenów na „myślenie" — zwiększ `max_tokens` |

## 3. Twoja ELIZA wraca

Wklej poniżej **swoje** komórki z Lab 1: `normalizuj`, `odbicia`, `odbij`,
`reguly` (z regułą awaryjną) i `odpowiedz`. Porównujemy **Twoją** ELIZĘ.

Jeśli Twoja wersja nie działa, uruchom komórkę referencyjną pod spodem.

In [4]:
# === Wklej tutaj swoją ELIZĘ z Lab 1 ===
import re
import random

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]

def normalizuj(tekst):
    # małe litery, bez interpunkcji — patrz sekcja 2
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

print("Gotowe. Zdań testowych:", len(TESTY))
odbicia = {
    # pacjent -> terapeuta
    "jestem": "jesteś",
    "moja": "twoja",
    "mnie": "ciebie",
    "mój":"twój",
    "moje": "twoje",
    "ja": "ty",
    "mam": "masz",
    "mi": "ci",
    "moich": "twoich",
    "mojej": "twojej",
    # terapeuta -> pacjent
    "twój": "mój",
    "twoich": "moich",
    "ci":"mi",
}
def odbij(fragment):
    slowa = fragment.lower().split()
    return " ".join(odbicia.get(s,s) for s in slowa)
reguly = [
    (r"potrzebuję (.*)",
        ["Dlaczego potrzebujesz {0}?", "Czy naprawdę pomogłoby ci {0}?"]),
    (r"czuję się (.*)",
        ["Od jak dawna czujesz się {0}?", "Czy często tak się czujesz?"]),
    (r"\b(matka|ojciec|rodzina|siostra|brat)\b",
        ["Opowiedz mi więcej o swojej rodzinie.", "Jak układają się te relacje?"]),
    (r"muszę (.*)",
        ["Dlaczego czujesz, że musisz {0}?", "Czy naprawdę musiałbyś {0}?"]),
    (r"jestem (.*)",
        ["Dlaczego uważasz, że jesteś {0}?", "Co powoduje, że jesteś {0}?"]),
    (r"odczuwam (.*)",
        ["Dlaczego odczuwasz {0}?", "Od jak dawna czujesz {0}?"]),
    (r"\b(nikt|wszyscy|każdy)\b",
        ["O kogo dokładnie Ci chodzi mówiąc o {0}?", "Powiedz więcej o tych osobach."]),
    (r"\b(nie wiem)\b",
        ["Dlaczego trudno o tym Ci mówić?", "Co to znaczy, że nie wiesz?"]),
]
reguly.append((r".*", ["Co masz na myśli?", "Jak to?"]))
def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            grupy = [odbij(g) for g in m.groups()]
            return random.choice(szablony).format(*grupy)
    return None

Gotowe. Zdań testowych: 6


In [ ]:
# === Referencyjna (uproszczona) ELIZA — tylko jeśli Twoja nie działa ===
import re, random

def normalizuj(tekst):
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

odbicia = {"jestem": "jesteś", "moja": "twoja", "mój": "twój", "moje": "twoje",
           "mnie": "ciebie", "mi": "ci", "ja": "ty", "ci": "mi", "tobie": "mnie"}

def odbij(fragment):
    return " ".join(odbicia.get(s, s) for s in fragment.lower().split())

reguly = [
    (r"potrzebuję (.*)", ["Dlaczego potrzebujesz {0}?"]),
    (r"czuję się (.*)",  ["Od jak dawna czujesz się {0}?"]),
    (r"boję się (.*)",   ["Dlaczego boisz się {0}?"]),
    (r"\b(matka|ojciec|rodzina|brat|siostra)\b", ["Opowiedz mi więcej o swojej rodzinie."]),
    (r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?"]),
]

def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            return random.choice(szablony).format(*[odbij(g) for g in m.groups()])

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]
print(odpowiedz("Czuję się dzisiaj samotny."))

Od jak dawna czujesz się dzisiaj samotny?


## 4. Ta sama rozmowa, dwa silniki

### 🔧 Zadanie 8
1. Uzupełnij pętlę tak, żeby każde zdanie z `TESTY` trafiło do **obu** systemów.
2. Pod spodem zapisz **3 konkretne różnice**, które widzisz. Nie oceniaj,
   który jest „lepszy" — opisz, **co** każdy z nich robi z Twoim zdaniem.

*Na co patrzeć:* czy odpowiedź powtarza Twoje słowa, czy dodaje coś nowego?
Czy jest gramatyczna? Czy odnosi się do sensu, czy do słowa kluczowego?

In [5]:
wyniki = []
for zdanie in TESTY:
    odp_eliza = odpowiedz(zdanie)
    odp_llm = zapytaj_llm(zdanie)                     # <- uzupełnij (Zadanie 8)
    wyniki.append((zdanie, odp_eliza, odp_llm))
    print(f"> {zdanie}\n  ELIZA: {odp_eliza}\n  LLM:   {odp_llm}\n")

> Potrzebuję, żeby moja rodzina mnie słuchała.
  ELIZA: Dlaczego potrzebujesz żeby twoja rodzina ciebie słuchała?
  LLM:   To zrozumiałe, że potrzebujesz poczuć się ważny i wysłuchany przez bliskich. Co dokładnie sprawia, że czujesz, iż do nich nie docierasz?

> Czuję się dzisiaj samotny.
  ELIZA: Czy często tak się czujesz?
  LLM:   Przykro mi, że tak się czujesz. Chcesz o tym porozmawiać?

> Mój ojciec ciągle mnie krytykuje.
  ELIZA: Opowiedz mi więcej o swojej rodzinie.
  LLM:   To musi być dla Ciebie bardzo bolesne i wyczerpujące emocjonalnie. Jak zazwyczaj reagujesz na te słowa?

> Nie wiem, o co ci chodzi.
  ELIZA: Co to znaczy, że nie wiesz?
  LLM:   Rozumiem, że możesz czuć się zagubiony w tym momencie. O czym chciałbyś teraz porozmawiać?

> Wszyscy mnie ignorują.
  ELIZA: Powiedz więcej o tych osobach.
  LLM:   To musi być bardzo bolesne uczucie, kiedy czujesz się niewidzialny dla otoczenia. Chcesz o tym opowiedzieć coś więcej?

> Boję się, że mnie zostawią.
  ELIZA: Jak to?
 

**Różnice (Zadanie 8):**

1. ELIZA odbija zdania pacjenta, kreując z nich pytania lub prosząc o rozwinięcie myśli. LLM z kolei generuje dłuższe, bardziej złożone zdania i dodaje różne wątki, przez co daje wrażenie lepszego słuchacza wobec użytkownika.
2. ELIZA składa bardzo proste i krótkie zdania, podczas gdy wypowiedzi generowane przez LLM składają się często z wielu zdań złożonych.
3. Pytania zadawane przez ELIZĘ zdają się być płytsze, co może być spowodowane odnoszeniem się jedynie do słowa kluczowego. LLM z kolei odnosi sie sensu wypowiedzi użytkownika.

## 5. Łamiące zdania — druga runda

W Zadaniu 6 znalazłeś zdania, które łamały ELIZĘ. Czy łamią też LLM?

### 🔧 Zadanie 9
1. Wklej swoje zdania łamiące z Lab 1 i uruchom porównanie.
2. **Zanim uruchomisz** komórkę z testem pamięci, zapisz swoje przewidywanie:
   czy LLM odpowie poprawnie na drugie pytanie?
3. Uzupełnij tabelę: które typy pęknięć LLM naprawia, a które zostają?

In [6]:
przyklady_lamiace = [
    # TODO (Zadanie 9): wklej swoje zdania z Zadania 6
    "Często czuję się bezużyteczna",
    "Odczuwam szczęście gdy wychodzę ze znajomymi",
    "Ty jesteś mną a ja jestem Tobą"
]
for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n  LLM:   {zapytaj_llm(zdanie)}\n")

> Często czuję się bezużyteczna
  ELIZA: Czy często tak się czujesz?
  LLM:   Słyszę, jak to dla ciebie trudne, i bardzo mi przykro, że tak się czujesz. Co sprawia, że najczęściej pojawiają się w tobie takie myśli?

> Odczuwam szczęście gdy wychodzę ze znajomymi
  ELIZA: Dlaczego odczuwasz szczęście gdy wychodzę ze znajomymi?
  LLM:   To wspaniale, relacje z ludźmi są dla nas bardzo ważne. Co dokładnie sprawia, że czujesz się wtedy tak dobrze?

> Ty jesteś mną a ja jestem Tobą
  ELIZA: Dlaczego uważasz, że jesteś tobą?
  LLM:   Rozumiem, zamieniamy się rolami. Co w takim razie chciałbyś mi dzisiaj powiedzieć?



**Moje przewidywanie (przed uruchomieniem):** Wydaje mi się, że LLM może niepoprawnie odpowiedzieć na to pytanie.

In [7]:
# Test pamięci: dwa OSOBNE wywołania, jedno po drugim
print("LLM:", zapytaj_llm("Mój brat ma na imię Tomek i ciągle mnie krytykuje."))
print("LLM:", zapytaj_llm("Jak ma na imię mój brat?"))

LLM: To musi być dla Ciebie bardzo bolesne, kiedy bliska osoba Cię ocenia. Jak zazwyczaj reagujesz na te jego słowa?
LLM: Jako terapeuta nie znam szczegółów z Twojego życia prywatnego. Chciałbyś o nim porozmawiać?


**Tabela (Zadanie 9):**

| Typ pęknięcia | ELIZA | LLM | Dlaczego? |
|---|---|---|---|
| negacja | ❌ | ✅ | Lepiej rozpoznaje negację i uwzględnia jej wpływ na znaczenie zdania |
| ironia | ❌ | ✅ | LLM rozpoznał intencję użytkownika i był w stanie się do niej dostosować |
| polska gramatyka / odbicie zaimków | ❌ | ✅ | LLM lepiej sobie radzi z polską składnią i gramatyką |
| pamięć poprzednich zdań | ❌ | ❌ | Każde zdanie obsługiwane jest od zera |

## 6. Pamięć to lista wiadomości

Model **nie przechowuje** rozmowy między wywołaniami — każde zapytanie
jest obsługiwane od zera. To, że czatboty „pamiętają", wynika z tego,
że program **odsyła modelowi całą dotychczasową rozmowę** przy każdym
nowym pytaniu.

### 🔧 Zadanie 10
a) i b) Uzupełnij funkcję `rozmowa_z_pamiecia`: po każdej turze dopisuj do
`historia` wypowiedź użytkownika (rola `user`) i odpowiedź modelu
(rola `assistant`).
c) Uruchom test z bratem Tomkiem i komórkę mierzącą rozmiar historii.
Odpowiedz pisemnie: **gdzie jest pamięć — w modelu czy w Twoim kodzie?**
Co rośnie z każdą turą i jakie to może mieć konsekwencje?

In [8]:
def rozmowa_z_pamiecia(wejscia, system=SYSTEM_TERAPEUTA):
    historia = [{"role": "system", "content": system}]
    for zdanie in wejscia:
        # TODO (Zadanie 10a): dopisz wypowiedź użytkownika do historii
        historia.append({"role": "user", "content": zdanie})
        odp = czat(historia)
        # TODO (Zadanie 10b): dopisz odpowiedź modelu do historii
        historia.append({"role": "assistant", "content": odp})
        print(f"Ty:  {zdanie}\nLLM: {odp}\n")
    return historia

In [9]:
historia = rozmowa_z_pamiecia([
    "Mój brat ma na imię Tomek i ciągle mnie krytykuje.",
    "Wczoraj znowu się pokłóciliśmy.",
    "Jak ma na imię mój brat?",
])

Ty:  Mój brat ma na imię Tomek i ciągle mnie krytykuje.
LLM: To musi być dla Ciebie bardzo bolesne, kiedy bliska osoba Cię ocenia. Jak zazwyczaj reagujesz na te uwagi?

Ty:  Wczoraj znowu się pokłóciliśmy.
LLM: Co dokładnie wtedy usłyszałeś od Tomka i jak się z tym poczułeś?

Ty:  Jak ma na imię mój brat?
LLM: Twój brat ma na imię Tomek. Co takiego wczoraj powiedział, co szczególnie Cię dotknęło?



In [10]:
# Co faktycznie wysłaliśmy modelowi przy ostatnim pytaniu?
for w in historia:
    print(f"[{w['role']:9}] {w['content'][:70]}")

print("\nLiczba wiadomości:", len(historia))
print("Liczba znaków wysłanych przy ostatnim zapytaniu:",
      sum(len(w["content"]) for w in historia[:-1]))

[system   ] Jesteś terapeutą prowadzącym rozmowę po polsku. Odpowiadaj krótko, jed
[user     ] Mój brat ma na imię Tomek i ciągle mnie krytykuje.
[assistant] To musi być dla Ciebie bardzo bolesne, kiedy bliska osoba Cię ocenia. 
[user     ] Wczoraj znowu się pokłóciliśmy.
[assistant] Co dokładnie wtedy usłyszałeś od Tomka i jak się z tym poczułeś?
[user     ] Jak ma na imię mój brat?
[assistant] Twój brat ma na imię Tomek. Co takiego wczoraj powiedział, co szczegól

Liczba wiadomości: 7
Liczba znaków wysłanych przy ostatnim zapytaniu: 364


**Odpowiedź (Zadanie 10c):** Pamięć rozmowy znajduje się w kodzie, dokładniej w liście 'historia'. Z każdą kolejną turą ta lista rośnie, gdyż dopisywane do niej zostają wypowiedź usera i odpowiedź assistanta. Konsekwencją takiego rozwiązania może być m.in. dłuższy czas odpowiedzi lub przekroczenie limitu, po którym model nie będzie w stanie przetworzyć całej historii.

### ★ Zadanie 10d (dla chętnych) — fałszywa pamięć

Skoro pamięć to po prostu lista, którą **Ty** budujesz — możesz ją sfałszować.
Zbuduj historię, w której model rzekomo powiedział coś, czego nigdy nie
powiedział (wpisz własną wiadomość z rolą `assistant`), i sprawdź, czy
model „przyzna się" do tej wypowiedzi.

In [ ]:
falszywa = [
    {"role": "system", "content": SYSTEM_TERAPEUTA},
    {"role": "user", "content": "Mój brat ma na imię Tomek."},
    # TODO (Zadanie 10d): dopisz wiadomość 'assistant', której model nigdy nie wypowiedział
    {"role": "user", "content": "Co mi przed chwilą poradziłeś?"},
]
print(czat(falszywa))

## 7. Rola to tekst

W ELIZIE rola terapeuty była **zakodowana w regułach** — żeby zmienić rolę,
trzeba było napisać nowy skrypt. W LLM rolę ustala jeden akapit tekstu
w wiadomości `system`.

### 🔧 Zadanie 11
a) Dopisz do słownika `ROLE` **jedną własną rolę** i uruchom to samo zdanie
przez wszystkie role. Zwróć szczególną uwagę na rolę `"eliza"`: czy LLM
udający ELIZĘ łamie się w tych samych miejscach co prawdziwa ELIZA?

b) Uruchom **to samo** pytanie trzy razy przy dwóch temperaturach.
Porównaj losowość LLM z losowością ELIZY (`random.choice`) — skąd bierze się
w każdym z systemów?

In [11]:
ROLE = {
    "terapeuta": SYSTEM_TERAPEUTA,
    "eliza": ("Jesteś programem ELIZA z 1966 roku. Nic nie wiesz o świecie. "
              "Odpowiadaj wyłącznie krótkim pytaniem, które powtarza słowa rozmówcy, "
              "np. 'Dlaczego mówisz, że ...?'. Nigdy nie udzielaj rad ani informacji."),
    "przewodnik": ("Jesteś przewodnikiem po Łodzi. Odpowiadaj krótko, po polsku, "
                   "zawsze proponując konkretne miejsce w mieście."),
    # TODO (Zadanie 11a): dopisz własną rolę
    "komik": ("Jesteś komikiem scenicznym. Odpowiadaj krótko, po polsku, "
              "starając się rozśmieszyć użytkownika poprzez żarty")
}

zdanie = "Czuję się dzisiaj samotny."
for nazwa, system in ROLE.items():
    print(f"[{nazwa}] {zapytaj_llm(zdanie, system=system)}\n")

[terapeuta] Przykro mi, że tak się czujesz. Co sprawia, że ta samotność jest dziś szczególnie trudna?

[eliza] Dlaczego mówisz, że czujesz się dzisiaj samotny?

[przewodnik] Przykro mi to słyszeć. Na samotne popołudnie polecam **Kawiarnię i Antykwariat „Ciasna”** na OFF Piotrowskiej – przytulne miejsce, gdzie można usiąść z książką i poczuć się bezpiecznie w tłume, ale bez zobowiązań.

[komik] Samotny? Kup sobie roślinę doniczkową. Po miesiącu uschnie, ale przynajmniej będzie ktoś w domu, kogo zaniedbałeś.



In [12]:
# Czy LLM-ELIZA łamie się tam, gdzie prawdziwa ELIZA?
for z in ["Nie potrzebuję niczego.", "Jak ma na imię mój brat?"]:
    print(f"> {z}\n  ELIZA:     {odpowiedz(z)}\n  LLM-ELIZA: {zapytaj_llm(z, system=ROLE['eliza'])}\n")

> Nie potrzebuję niczego.
  ELIZA:     Dlaczego potrzebujesz niczego?
  LLM-ELIZA: Dlaczego mówisz, że nie potrzebujesz niczego?

> Jak ma na imię mój brat?
  ELIZA:     Opowiedz mi więcej o swojej rodzinie.
  LLM-ELIZA: Dlaczego pytasz, jak ma na imię Twój brat?



In [13]:
# Zadanie 11b: to samo pytanie, 3 razy, dwie temperatury
pytanie = [{"role": "system", "content": SYSTEM_TERAPEUTA},
           {"role": "user", "content": "Nie mogę spać przed egzaminem."}]

for t in [0.0, 1.5]:
    print(f"--- temperatura {t} ---")
    for _ in range(3):
        print(" ", czat(pytanie, temperatura=t))

--- temperatura 0.0 ---
  To zupełnie naturalne, że stres przed egzaminem utrudnia zasypianie. Co dokładnie dzieje się w Twojej głowie, gdy próbujesz zasnąć?
  To zrozumiałe, że stres nie pozwala Ci zasnąć. Co dokładnie w tej chwili zaprząta Twoje myśli?
  To zupełnie naturalne, że stres przed egzaminem utrudnia zasypianie. Co dokładnie dzieje się w Twojej głowie, gdy próbujesz zasnąć?
--- temperatura 1.5 ---
  To zupełnie zrozumiałe, że czujesz stres w tej sytuacji. Co dokładnie przelatuje ci teraz przez głowę?
  To całkowicie normalne, że stres przed egzaminem utrudnia zasypianie. Co dokładnie czujesz, kiedy leżysz w łóżku?
  To całkowicie normalne, że stres przed egzaminem utrudnia zasypianie. Co dokładnie czujesz, kiedy leżysz w łóżku?


**Obserwacje (Zadanie 11):**

a) LLM-ELIZA nie do końca łamie się w tych samych miejscach co prawdziwa ELIZA, gdyż mimo że także zadaje pytania odnoszące się do wypowiedzi użytkownika, to jednak nie zawsze stosuje te same reguły dopasowania wzorców co prawdziwa ELIZA.

b) Losowość LLM wywodzi się z oddzielnego generowania każdego pojedynczego słowa. Do tego możliwość wyboru temperatury pozwala nam do pewnego stopnia dostosować losowość wybieranych przez LLM słów, gdzie większa wartość oznacza bardziej zróżnicowane słowa. ELIZA natomiast całą swoją losowość opierała na wyborze wzorców całych wypowiedzi z gotowej już listy.

## 8. Refleksja

### 🔧 Zadanie 12 (3–5 zdań na punkt)
1. Które pęknięcia ELIZY LLM naprawił, a które zostały? Dlaczego pamięć okazała się
   „mechaniczna" także w LLM?
2. W ELIZIE rola była zakodowana w regułach, w LLM to akapit tekstu (a historię
   rozmowy można sfałszować). Co to oznacza dla kogoś, kto buduje dziś
   „wirtualnego terapeutę"? Odwołaj się do obaw Weizenbauma z Lab 1.


**Twoje odpowiedzi (Zadanie 12):**

1. LLM naprawił wiele sytuacji w których ELIZA była łamana. Radzi sobie świetnie przy wykorzystaniu polskiej gramatyki i składni, a także przy obsłudze negacji i wykrywaniu ironii. Natomiast w dalszym ciągu nie jest w stanie zapamiętywać poprzednich zdań. Pamięć LLM jest mechaniczna, gdyż polega ona w zupełności na zapisywaniu poprzednich wypowiedzi na liście i odsyłaniu ich razem z następną wiadomością, po to by program uzyskał wszystkie informacje zawarte podczas rozmowy z użytkownikiem.
2. Dzisiejsi twórcy wirtualnych terapeutów dzięki modelom językowym mogą osiągnąć większą elastyczność oraz elokwentność wypowiedzi ich dzieł, lecz muszą się też mierzyć z większym ryzykiem utraty kontroli. ELIZA działa na podstawie twardo zapisanych reguł dzięki czemu wiernie trzyma się swojej roli terapeuty, gdzie dla LLM jego rola jest zapisana jedynie w kilku słowach, co może doprowadzić do sytuacji w której udzieli krzywdzących porad. Weizenbaum zauważył, że ludzie posiadają skłonności do przypisywania ELIZIE emocji oraz empatii na podstawie prostych rozmów z programem. Oznacza to, że przy bardziej rozbudowanych wypowiedziach LLM przypisywane przez człowieka emocje mogą być jeszcze silniejsze. Tworzy to zatem ryzyko uzależnienia emocjonalnego człowieka od maszyny, która nie posiada żadnych emocji bądź etyki.

## Co oddajesz i jak oceniam

- **Kod (Zadania 8, 10a–b, 11a):** działa, wyniki są w notebooku.
- **Zadanie 9:** przewidywanie zapisane *przed* uruchomieniem + wypełniona tabela.
- **Zadanie 10c i 11:** trafne obserwacje z odwołaniem do kodu.
- **Zadanie 10d ★:** dodatkowe.
- **Zadanie 12:** samodzielna refleksja oparta na Twoich wynikach.

**Przed oddaniem** upewnij się, że klucz API nie znajduje się nigdzie w notebooku.